**Anomalib**

Deep Learning Library which has SOTA anomaly detection algorithms

In [58]:
pip install anomalib


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [59]:
from anomalib.data import MVTecAD

categories = ["bottle", "hazelnut","transistor"]
datamodules = {}
train_loaders = {}

for cat in categories:
    datamodule = MVTecAD(
        root="./datasets/MVTecAD",
        category=cat,
        train_batch_size=32,
        eval_batch_size=32
    )
    
    datamodule.prepare_data()
    datamodule.setup()
    
    datamodules[cat] = datamodule
    train_loaders[cat] = datamodule.train_dataloader()

In [60]:
import os
import glob
import numpy as np
from PIL import Image
from sklearn.metrics import roc_auc_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader
from torchvision import models,transforms
from torchvision.models import ResNet50_Weights

## Data Loading and Processing

In [61]:
class MVTecDataset(Dataset):
    def __init__(self,root_dir,category,is_train=True,transform=None,mask_transform=None):
        self.transform = transform
        self.mask_transform = mask_transform
        self.image_paths = []
        self.mask_paths = []
        self.labels = []
        
        if is_train:
            img_dir = os.path.join(root_dir,category,"train","good")
            paths = glob.glob(os.path.join(img_dir,"*.png"))
            self.image_paths.extend(paths)
            self.labels.extend([0]*len(paths))
            self.mask_paths.extend([None]*len(paths))
        
        else:
            test_dir = os.path.join(root_dir,category,"test")
            for defect in os.listdir(test_dir):
                if defect.startswith('.'):
                    continue
                paths = glob.glob(os.path.join(test_dir,defect,"*.png"))
                self.image_paths.extend(paths)
                
                if defect == "good":
                    self.labels.extend([0]*len(paths))
                    self.mask_paths.extend([None]*len(paths))
                else:
                    self.labels.extend([1]*len(paths))
                    for p in paths:
                        mask_path = os.path.join(root_dir,category,"ground_truth",defect,os.path.basename(p).replace(".png","_mask.png"))
                        self.mask_paths.append(mask_path)
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self,idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        label = self.labels[idx]
        mask_path = self.mask_paths[idx]
        
        if mask_path and os.path.exists(mask_path):
            mask = Image.open(mask_path).convert("L")
            if self.mask_transform:
                mask = self.mask_transform(mask)
        else:
            mask = torch.zeros((1,image.shape[1],image.shape[2]))
        return image,label,mask

## Feature Extraction from ResNet Layers

Features from Layer 2 and 3 of the ResNet are extracted : which are neither too coarse nor too fine to work with (hence did not choose Layer 1 or 4)


Layer 2 : 512 Channels

Layer 3 : 1024 Channels

But Layer 3 image is of smaller size than Layer 2, so up-scaled using billinear interpolation

In [62]:
class ResNetFeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
        
        # 512 Channels
        self.encoder_l2 = nn.Sequential(
            resnet.conv1,resnet.bn1,resnet.relu,resnet.maxpool,resnet.layer1,resnet.layer2
        )
        
        # 1024 Channels
        self.encoder_l3 = resnet.layer3
        
        for param in self.parameters():
            param.requires_grad = False
    
    def forward(self,x):
        with torch.no_grad():
            feat2 = self.encoder_l2(x) # Larger Image
            feat3 = self.encoder_l3(feat2) # Smaller Image : Needs to be resized before concat
            
            feat3_up = nn.functional.interpolate(feat3,size=feat2.shape[2:],mode="bilinear",align_corners=False)
            return torch.cat([feat2,feat3_up],dim=1)

## 1x1 Convolutional Autoencoder

1536 -> 512 -> 128 -> 512 -> 1536 network which needs to reconstruct image from the features

The 128 dimensional bottleneck before expanding back to 1536 against a MSE Loss forces network to learn the core essential patterns 

Trained only on train/good so that model learns to encode and decode perfect textures

When OOD image is passed, the bottleneck will fail to compress to 128 features and reconstruct from it, as a result MSE between input and reconstrcuted image will be very large

*Similar idea as 1_MSP_Baseline in OOD*

In [63]:
class ConvAutoEncoder(nn.Module):
    def __init__(self,in_channels=1536):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels=in_channels,out_channels=512,kernel_size=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels=512,out_channels=128,kernel_size=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True)
        )
        self.decoder = nn.Sequential(
            nn.Conv2d(in_channels=128,out_channels=512,kernel_size=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels=512,out_channels=in_channels,kernel_size=1)
        )
    
    def forward(self,x):
        return self.decoder(self.encoder(x))

In [ ]:
def train_evaluate(category="bottle",root_dir="./data/mvtec_ad",epochs=20,batch_size=32):
    device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
    
    img_size = 256
    transform = transforms.Compose([
        transforms.Resize((img_size,img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    mask_transform = transforms.Compose([
        transforms.Resize((img_size,img_size)),
        transforms.ToTensor()
    ])
    
    train_loader = DataLoader(MVTecDataset(root_dir,category,is_train=True,transform=transform),batch_size=batch_size,shuffle=True)
    test_loader = DataLoader(MVTecDataset(root_dir,category,is_train=False,transform=transform,mask_transform=mask_transform),batch_size=1,shuffle=False)
    
    extractor = ResNetFeatureExtractor().to(device).eval()
    autoencoder = ConvAutoEncoder().to(device)
    optimizer = torch.optim.Adam(autoencoder.parameters(),lr=1e-3)
    criterion = nn.MSELoss()
    
    print(f"Autoencoder training for {category}")
    autoencoder.train()
    for epoch in range(epochs):
        epoch_loss = 0
        for images,labels,masks in train_loader:
            images = images.to(device)
            features = extractor(images)
            
            optimizer.zero_grad()
            reconstr = autoencoder(features)
            loss = criterion(reconstr,features)
            loss.backward()
            optimizer.step()
            
            epoch_loss+=loss.item()
            
        print(f"Epoch {epoch+1}/{epochs}  |  MSE Loss : {epoch_loss/len(train_loader):.4f}")
    
    print(f"Evaluating {category}")
    autoencoder.eval()
    
    image_scores,image_labels = [],[]
    pixel_scores,pixel_labels = [],[]

    with torch.no_grad():
        for images,labels,masks in test_loader:
            images = images.to(device)
            features = extractor(images)
            reconstr = autoencoder(features)
            
            mse_map = torch.mean((features-reconstr)**2,dim=1,keepdim=True)
            
            # Ground_Truth is 256x256 but mse_map is only 32x32
            anomaly_map = nn.functional.interpolate(mse_map,size=(img_size,img_size),mode="bilinear",align_corners=False)
            anomaly_map = anomaly_map.cpu().squeeze().numpy()
            mask = masks.squeeze().numpy()
            
            image_scores.append(np.max(anomaly_map))
            image_labels.append(labels.item())
            
            
            pixel_scores.extend(anomaly_map.flatten())
            pixel_labels.extend((mask.flatten()>0).astype(int))
            
    img_auroc = roc_auc_score(image_labels,image_scores)
    pxl_auroc = roc_auc_score(pixel_labels,pixel_scores)
    
    print(f"[{category.upper()}] Image AUROC : {img_auroc:.4f}  |  Pixel AUROC : {pxl_auroc:.4f}")        

## Performance

In [65]:
if __name__ == "__main__":
    target_categories = ["bottle","hazelnut","transistor"]
    
    for cat in target_categories:
        print("\n" + "-"*50)
        print(f"Starting Pipeline for : {cat.upper()}")
        print("-"*50)
        
        train_evaluate(category=cat,root_dir="./datasets/MVTecAD",epochs=15)
        print()
        print()



--------------------------------------------------
Starting Pipeline for : BOTTLE
--------------------------------------------------
Autoencoder training for bottle
Epoch 1/15  |  MSE Loss : 0.0888
Epoch 2/15  |  MSE Loss : 0.0245
Epoch 3/15  |  MSE Loss : 0.0137
Epoch 4/15  |  MSE Loss : 0.0098
Epoch 5/15  |  MSE Loss : 0.0080
Epoch 6/15  |  MSE Loss : 0.0071
Epoch 7/15  |  MSE Loss : 0.0065
Epoch 8/15  |  MSE Loss : 0.0060
Epoch 9/15  |  MSE Loss : 0.0057
Epoch 10/15  |  MSE Loss : 0.0055
Epoch 11/15  |  MSE Loss : 0.0052
Epoch 12/15  |  MSE Loss : 0.0050
Epoch 13/15  |  MSE Loss : 0.0049
Epoch 14/15  |  MSE Loss : 0.0047
Epoch 15/15  |  MSE Loss : 0.0045
Evaluating bottle
[BOTTLE] Image AUROC : 0.9452  |  Pixel AUROC : 0.9565



--------------------------------------------------
Starting Pipeline for : HAZELNUT
--------------------------------------------------
Autoencoder training for hazelnut
Epoch 1/15  |  MSE Loss : 0.0568
Epoch 2/15  |  MSE Loss : 0.0108
Epoch 3/15  |  MSE Los

As expected, the model performs very well on easier tasks like bottle and hazelnut but does poorly on harder tasks like transistor